In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
pio.templates.default = "plotly_white"
import xarray as xr
from whakaaribn.visualize import trellis_plot
from whakaaribn import get_color

/usr/local/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
try:
    grid_search_results = snakemake.input.grid_search_results
except NameError as e:
    grid_search_results = '../results/grid_search_results.csv'

In [3]:
search_results = pd.read_csv(grid_search_results, converters={"params": eval})

In [ ]:
fig = go.Figure()
metric = 'mod_roc_auc_no_pew'
nstates = sorted(search_results.param_clf__nstates.unique())
pews = sorted(search_results.param_clf__pew.unique())
for _nstates in nstates: 
    vals = []
    error = []
    showlegend=True
    for pew in pews:
        sdf_tmp = search_results.loc[(search_results.param_clf__pew == pew) & (search_results.param_clf__nstates == _nstates)]
        sdf_tmp = sdf_tmp.sort_values(by=[f'rank_test_{metric}'])
        vals.append(sdf_tmp.iloc[0][f'mean_test_{metric}'])
        error.append(sdf_tmp.iloc[0][f'std_test_{metric}'])
    vals = np.array(vals)
    error = np.array(error)
    fig.add_trace(go.Scatter(x=list(pews), y=vals, mode='lines',
                             name=f'{_nstates} states', line_color=get_color(_nstates-3)))
    fig.add_trace(go.Scatter(x=list(pews), y=vals+error, mode='lines', marker=dict(color="#444"),
                                 line=dict(width=0), showlegend=False))
    fig.add_trace(go.Scatter(x=list(pews), y=vals-error, mode='lines', marker=dict(color="#444"),
                                 line=dict(width=0), showlegend=False, fillcolor=get_color(_nstates-3, alpha=0.1),
                                 fill='tonexty'))
 
fig.update_layout(xaxis_title="Pre-eruption window", yaxis_title='AUC-ROC')
try:
    fig.write_image(snakemake.output.model_objective_function, width=900, height=400, scale=5)
except NameError:
    pass
fig